# 🏥 HealthConnect Appointment Analysis

## Week 5: Exploratory Analysis, KPI Development & Business Insights

**AnalystLab Africa — HealthConnect Experience Lab**

**Data Analytics Track**

Prepared by **Hudu Yusuf Ibrahim**

----

## Week 4 Foundation Review

Before beginning Week 5's practical analysis, this section briefly reviews the foundation established in Week 4. It is not a repetition of the full Week 4 work, but a short recap to provide context for the analysis that follows.

- **Problem defined:** Investigating which patient, booking, reminder, and logistical factors are associated with appointment no-shows at HealthConnect Clinic.
- **Proposed approach:** Validate the dataset, investigate the identified business questions, calculate and assess the candidate KPIs against appointment outcomes, and use the findings to inform an initial Power BI dashboard.
- **Resources used:** `HealthConnect_Appointment_Data.csv` and `HealthConnect_Data_Dictionary.csv`.
- **Key assumptions/limitations:** The dataset is synthetic, with a 48.5% no-show rate that should not be interpreted as representative of real clinics. `previous_appointments` and `previous_no_shows` may not reflect coherent real patient histories. Small amounts of missing data remain in `distance_to_clinic_km` and `waiting_time_minutes`.
- **Week 5 focus:** Validate the dataset, investigate the identified business questions, calculate and assess the candidate KPIs, identify meaningful patterns, and develop evidence-based business insights and recommendations.


----

## Data Loading 

In [1]:
import pandas as pd

df = pd.read_csv("HealthConnect_Appointment_Data.csv")
df['booking_date'] = pd.to_datetime(df['booking_date'])
df['appointment_date'] = pd.to_datetime(df['appointment_date'])

df.shape

(5000, 18)

### Observation

The dataset contains **5,000 appointment records and 18 columns**, confirming that the dataset structure is unchanged from the Week 4 assessment.

The `booking_date` and `appointment_date` columns were also converted from text to **datetime format**, which will allow accurate date calculations and validation during the analysis.

---

## Data Preparation / Inconsistent Values

### Categorical Check

In [ ]:
categorical_columns = [
    'gender',
    'age_group',
    'appointment_type',
    'appointment_day',
    'reminder_sent',
    'reminder_channel',
    'appointment_outcome'
]

for col in categorical_columns:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

### Observation

No inconsistent values were found across the checked categorical columns — all categories are clean, consistently labelled, and free of obvious typos or casing mismatches. The `NaN` values in `reminder_channel` are the previously confirmed structural missingness associated with records where no reminder was sent.

One point worth noting is that `appointment_day` contains appointments across all seven days, including Sunday (737 records, the highest single-day count). This would be unusual for a typical clinic operating schedule and is therefore treated as a characteristic of the synthetic dataset rather than a data quality defect.

### Numeric Check

In [3]:
numeric_columns = [
    'age',
    'booking_lead_days',
    'previous_appointments',
    'previous_no_shows',
    'distance_to_clinic_km',
    'waiting_time_minutes'
]

df[numeric_columns].describe()

,age,booking_lead_days,previous_appointments,previous_no_shows,distance_to_clinic_km,waiting_time_minutes
count,5000.000000,5000.00000,5000.000000,5000.000000,4910.000000,4940.000000
mean,48.794800,29.63860,3.013800,0.544200,10.109572,24.189676
std,18.138547,17.39936,1.741211,0.746832,6.590030,10.863184
min,18.000000,0.00000,0.000000,0.000000,0.500000,2.000000
25%,33.000000,15.00000,2.000000,0.000000,5.300000,17.000000
50%,49.000000,30.00000,3.000000,0.000000,8.700000,24.000000
75%,64.000000,45.00000,4.000000,1.000000,13.500000,32.000000
max,80.000000,60.00000,11.000000,5.000000,45.000000,68.000000


### Observation

The numerical variables fall within the expected ranges, with no obvious invalid or impossible values identified.

- `age` ranges from 18 to 80 years.
- `booking_lead_days` ranges from 0 to 60 days.
- `previous_appointments` ranges from 0 to 11.
- `previous_no_shows` ranges from 0 to 5.
- `distance_to_clinic_km` ranges from 0.5 to 45 km among non-missing records.
- `waiting_time_minutes` ranges from 2 to 68 minutes among non-missing records.

These ranges are consistent with the dataset definitions and previously identified constraints, so no numerical-value corrections are required at this stage.

### age_group validation

In [6]:
age_group_check = pd.cut(
    df['age'],
    bins=[17, 24, 34, 44, 54, 64, 80],
    labels=['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
)

(df['age_group'] != age_group_check).sum()

0

### Observation

The `age_group` field was independently recreated from the `age` variable and compared with the existing `age_group` column.

The comparison returned **0 mismatches**, confirming that `age_group` is correctly derived from `age` across all 5,000 records.

No correction is required for this field.

### appointment_day validation

In [7]:
appointment_day_check = df['appointment_date'].dt.day_name()

(df['appointment_day'] != appointment_day_check).sum()

0

### Observation

The `appointment_day` field was independently derived from `appointment_date` using the weekday name and compared with the existing `appointment_day` column.

The comparison returned **0 mismatches**, confirming that `appointment_day` is correctly derived from `appointment_date` across all 5,000 records.

No correction is required for this field.

### booking_lead_days validation

In [8]:
calculated_lead_days = (
    df['appointment_date'] - df['booking_date']
).dt.days

(df['booking_lead_days'] != calculated_lead_days).sum()

0

### Observation

The `booking_lead_days` field was independently validated by calculating the difference between `appointment_date` and `booking_date`.

The comparison returned **0 mismatches**, confirming that `booking_lead_days` correctly represents the number of days between booking and appointment dates across all 5,000 records.

No correction is required for this field, so it can be used confidently in the booking lead-time analysis.

---

## Exploratory Data Analysis (EDA)

Exploratory data analysis is used to understand patterns, distributions, and relationships within the HealthConnect appointment data.

The analysis will focus on appointment outcomes, patient characteristics, appointment history, reminders, booking lead time, distance, waiting time, and other factors relevant to appointment attendance and no-shows.

### reminders vs. attendance

In [13]:
pd.crosstab(df['reminder_sent'], df['appointment_outcome'], normalize='index') * 100

appointment_outcome,Attended,Cancelled,No-Show
reminder_sent,,,
No,42.679356,5.929722,51.390922
Yes,47.633462,5.008255,47.358283


### Observation

The outcome distribution differs by reminder status.

Among appointments where **no reminder was sent**, 51.4% resulted in a No-Show and 42.7% were Attended.

Among appointments where a **reminder was sent**, 47.4% resulted in a No-Show and 47.6% were Attended.

The No-Show proportion is therefore approximately **4.0 percentage points lower** among appointments that received a reminder, while the Attended proportion is approximately **4.9 percentage points higher**.

This suggests that reminder status is associated with appointment attendance in this dataset. However, this is a descriptive comparison and does not establish that reminders caused the difference.

## Reminder Channel

In [16]:
attended_reminder = df[df['reminder_sent'] == 'Yes']
pd.crosstab(attended_reminder['reminder_channel'], attended_reminder['appointment_outcome'], normalize='index') * 100

appointment_outcome,Attended,Cancelled,No-Show
reminder_channel,,,
Email,46.529081,5.065666,48.405253
SMS,49.600000,4.650000,45.750000
WhatsApp,44.595822,5.631244,49.772934


### Observation

Among appointments that received a reminder, the observed No-Show proportion differs by reminder channel.

- **SMS:** 45.8% No-Show
- **Email:** 48.4% No-Show
- **WhatsApp:** 49.8% No-Show

SMS has the lowest observed No-Show proportion, while WhatsApp has the highest. The difference between the lowest and highest channel is approximately **4.0 percentage points**.

This suggests that reminder channel may be associated with appointment outcomes in this dataset. However, these results are descriptive and do not establish that one reminder channel causes better attendance than another.

## booking_lead_days

In [17]:
df['lead_time_band'] = pd.cut(
    df['booking_lead_days'],
    bins=[-1, 7, 21, 60],
    labels=['Same week (0-7 days)', '2-3 weeks (8-21 days)', '3+ weeks (22-60 days)']
)

pd.crosstab(df['lead_time_band'], df['appointment_outcome'], normalize='index') * 100

appointment_outcome,Attended,Cancelled,No-Show
lead_time_band,,,
Same week (0-7 days),66.562500,5.625000,27.812500
2-3 weeks (8-21 days),58.139535,4.651163,37.209302
3+ weeks (22-60 days),37.642586,5.418251,56.939163


### Observation

Appointment outcome varies substantially across booking lead-time bands.

- **Same week (0–7 days):** 27.8% No-Show and 66.6% Attended.
- **2–3 weeks (8–21 days):** 37.2% No-Show and 58.1% Attended.
- **3+ weeks (22–60 days):** 56.9% No-Show and 37.6% Attended.

The No-Show proportion increases consistently as booking lead time becomes longer, rising from **27.8%** for same-week bookings to **56.9%** for appointments booked 22 days or more in advance. This represents an increase of approximately **29.1 percentage points**.

This indicates a strong association between longer booking lead time and No-Show outcomes in the dataset. However, the relationship is descriptive and does not establish that longer lead times directly cause patients to miss appointments.

## Previous No-Show History

In [18]:
df['previous_no_show_group'] = df['previous_no_shows'].apply(
    lambda x: 'Previous No-Show' if x > 0 else 'No Previous No-Show'
)

pd.crosstab(
    df['previous_no_show_group'],
    df['appointment_outcome'],
    normalize='index'
).mul(100).round(1)

appointment_outcome,Attended,Cancelled,No-Show
previous_no_show_group,,,
No Previous No-Show,50.5,6.0,43.5
Previous No-Show,40.4,4.2,55.4


### Observation

Appointment outcomes differ between patients with and without a previous No-Show history.

Patients with **previous no-shows** had a No-Show proportion of **55.4%**, compared with **43.5%** among patients with no previous No-Show history.

This represents an approximately **11.9 percentage-point higher No-Show proportion** among patients with previous no-shows.

The result suggests that previous No-Show history is associated with a higher likelihood of another No-Show in this dataset. This makes previous No-Show history a potentially useful indicator for identifying appointments that may require additional attention.

However, the finding is descriptive and does not establish that previous No-Show behaviour causes future No-Shows. The synthetic nature of the historical fields should also be considered when interpreting this relationship.

## Distance to the Clinic

In [19]:
df['distance_band'] = pd.cut(
    df['distance_to_clinic_km'],
    bins=[0, 5, 15, 45],
    labels=['<5km', '5-15km', '15km+']
)

pd.crosstab(df['distance_band'], df['appointment_outcome'], normalize='index').mul(100).round(1)

appointment_outcome,Attended,Cancelled,No-Show
distance_band,,,
<5km,49.0,4.6,46.5
5-15km,47.0,5.7,47.3
15km+,41.0,4.9,54.1


### Observation

Appointment outcomes show some variation across distance-to-clinic bands.

- **<5 km:** 46.5% No-Show and 49.0% Attended.
- **5–15 km:** 47.3% No-Show and 47.0% Attended.
- **15 km+:** 54.1% No-Show and 41.0% Attended.

The No-Show proportion increases from **46.5%** among appointments within 5 km to **54.1%** among appointments 15 km or more away, representing a difference of approximately **7.6 percentage points**.

This suggests that greater distance to the clinic is associated with a higher No-Show proportion in this dataset. However, the relationship is descriptive and does not establish that distance directly causes patients to miss appointments.

## Appointment Type vs. No-Show Rate

To assess whether appointment type is associated with appointment No-Shows, the No-Show Rate is calculated for each appointment type.

This extends the Week 4 analysis by examining appointment type directly against the target outcome rather than only reviewing the distribution of appointment types.

In [30]:
appointment_type_no_show = (
    pd.crosstab(
        df['appointment_type'],
        df['appointment_outcome'],
        normalize='index'
    )['No-Show']
    .mul(100)
    .round(1)
)

appointment_type_no_show

appointment_type
Diagnostic Test            49.7
Follow-up                  51.2
General Consultation       46.6
Specialist Consultation    47.4
Name: No-Show, dtype: float64

### Observation

No-Show rates vary relatively little across appointment types:

- **Follow-up:** 51.2%
- **Diagnostic Test:** 49.7%
- **Specialist Consultation:** 47.4%
- **General Consultation:** 46.6%

The difference between the highest and lowest groups is **4.6 percentage points**.

Compared with the stronger patterns observed for booking lead time and previous No-Show history, appointment type shows a relatively small difference in this dataset.

Therefore, appointment type does not appear to be a major distinguishing factor in No-Show rates based on this descriptive analysis. It is retained as a dashboard slicer so users can explore whether other patterns change across appointment types.

## Gender vs. No-Show Rate

To determine whether No-Show rates differ across gender groups, the No-Show Rate is calculated for each gender category.

This provides an additional descriptive comparison and supports the use of Gender as an interactive dashboard filter.

In [31]:
gender_no_show = (
    pd.crosstab(
        df['gender'],
        df['appointment_outcome'],
        normalize='index'
    )['No-Show']
    .mul(100)
    .round(1)
)

gender_no_show

gender
Female               48.4
Male                 48.7
Prefer not to say    43.5
Name: No-Show, dtype: float64

### Observation

No-Show rates for Female and Male patients are very similar:

- **Female:** 48.4%
- **Male:** 48.7%

The **Prefer not to say** group has a lower observed No-Show rate of 43.5%. However, this group contains only **108 records (2.2% of the dataset)**, so the difference should be interpreted cautiously.

Overall, Gender shows very little variation in No-Show rates in this dataset and does not appear to be a strong distinguishing factor based on this descriptive analysis.

Gender is retained as a dashboard slicer to allow users to explore the other No-Show patterns across gender groups.

**Neither appointment type nor gender appears to be a strong distinguishing factor in No-Show rates in this dataset.**

---

## KPI Analysis

## KPI 1: Overall No-Show Rate

**Definition:**  
The percentage of all scheduled appointments that resulted in a No-Show.

**Why it matters:**  
This provides the baseline measure of the appointment attendance problem.

**Calculation:**  
(No-Show appointments ÷ Total appointments) × 100

**Linked business question:**  
Q1 — What factors are most associated with appointment no-shows?

In [20]:
overall_no_show_rate = (df['appointment_outcome'] == 'No-Show').mean() * 100
overall_no_show_rate.round(1)

48.5

### Result and Interpretation

**Result:** **48.5%**

The overall No-Show rate is 48.5%, meaning nearly half of the appointments in the dataset resulted in a No-Show. This provides the baseline against which the remaining KPIs can be compared.

Because the dataset is synthetic, this figure should be interpreted as a characteristic of the dataset rather than as a representative real-world clinic attendance rate.

## KPI 2: No-Show Rate by Reminder Status / Channel

**Definition:**  
The percentage of appointments resulting in a No-Show, compared across reminder status and reminder channel.

**Why it matters:**  
This KPI helps assess whether appointment reminders are associated with attendance and whether the communication channel used may be relevant to the clinic's reminder strategy.

**Linked business question:**  
Q2 — Does sending a reminder relate to attendance, and does the reminder channel used make a difference?

In [21]:
kpi2_reminder_status = pd.crosstab(df['reminder_sent'], df['appointment_outcome'], normalize='index')['No-Show'].mul(100).round(1)
kpi2_reminder_status

reminder_sent
No     51.4
Yes    47.4
Name: No-Show, dtype: float64

### Result and Interpretation

**Result:**

- **No reminder:** 51.4% No-Show
- **Reminder sent:** 47.4% No-Show

Appointments where a reminder was sent had an observed No-Show rate approximately **4.0 percentage points lower** than appointments where no reminder was sent.

This indicates an association between reminder status and appointment outcome in the dataset. However, this does not establish that sending a reminder directly causes patients to attend, as other factors may also differ between the two groups.

### Reminder Channel

In [23]:
kpi2_reminder_channel = pd.crosstab(
    df[df['reminder_sent'] == 'Yes']['reminder_channel'],
    df[df['reminder_sent'] == 'Yes']['appointment_outcome'],
    normalize='index'
)['No-Show'].mul(100).round(1)

kpi2_reminder_channel

reminder_channel
Email       48.4
SMS         45.8
WhatsApp    49.8
Name: No-Show, dtype: float64

### Reminder Channel — Result and Interpretation

**Result:**

- **Email:** 48.4% No-Show
- **SMS:** 45.8% No-Show
- **WhatsApp:** 49.8% No-Show

Among appointments that received a reminder, **SMS had the lowest observed No-Show rate at 45.8%**, while **WhatsApp had the highest at 49.8%**.

The results show some variation in No-Show rates across reminder channels. However, the differences are relatively small, and the analysis is descriptive. Therefore, the results should not be interpreted as evidence that one reminder channel directly causes better attendance.

The reminder-status comparison also showed a lower No-Show rate among appointments where a reminder was sent (**47.4%**) compared with those where no reminder was sent (**51.4%**). Further analysis would be needed to determine whether other factors explain part of this difference.

## KPI 3: No-Show Rate by Booking Lead Time Band

**Definition:**  
The percentage of appointments resulting in a No-Show across different booking lead-time bands.

**Why it matters:**  
This KPI examines whether appointments booked further in advance are associated with a higher likelihood of No-Show.

**Linked business question:**  
Q3 — Does booking lead time affect the likelihood of a No-Show?

In [24]:
kpi3_lead_time = pd.crosstab(
    df['lead_time_band'],
    df['appointment_outcome'],
    normalize='index'
)['No-Show'].mul(100).round(1)

kpi3_lead_time

lead_time_band
Same week (0-7 days)     27.8
2-3 weeks (8-21 days)    37.2
3+ weeks (22-60 days)    56.9
Name: No-Show, dtype: float64

### Result and Interpretation

**Result:**

- **Same week (0–7 days):** 27.8% No-Show
- **2–3 weeks (8–21 days):** 37.2% No-Show
- **3+ weeks (22–60 days):** 56.9% No-Show

The No-Show rate increases consistently as booking lead time becomes longer. Appointments booked **3+ weeks in advance had a 56.9% No-Show rate**, compared with **27.8%** for appointments booked within the same week.

This represents a **29.1 percentage-point difference** between the two groups and is the largest difference observed among the booking lead-time bands.

The result suggests a strong association between longer booking lead time and No-Show rates in this dataset. From an operational perspective, appointments booked further in advance may benefit from stronger reminder or confirmation strategies closer to the appointment date.

However, this analysis is descriptive and does not establish that longer booking lead time causes patients to miss appointments.

## KPI 4: No-Show Rate Among Patients With Previous No-Shows

**Definition:**  
The percentage of appointments resulting in a No-Show among patients with at least one previous No-Show, compared with patients who have no previous No-Show history.

**Why it matters:**  
This KPI tests whether previous No-Show history is associated with a higher likelihood of another No-Show and can help identify whether targeted follow-up may be relevant.

**Linked business question:**  
Q4 — Are patients with a history of previous No-Shows more likely to miss future appointments?

In [25]:
kpi4_previous_no_show = pd.crosstab(
    df['previous_no_show_group'],
    df['appointment_outcome'],
    normalize='index'
)['No-Show'].mul(100).round(1)

kpi4_previous_no_show

previous_no_show_group
No Previous No-Show    43.5
Previous No-Show       55.4
Name: No-Show, dtype: float64

### Result and Interpretation

**Result:**

- **No Previous No-Show:** 43.5% No-Show
- **Previous No-Show:** 55.4% No-Show

Appointments involving patients with a previous No-Show had a **55.4% No-Show rate**, compared with **43.5%** among patients with no previous No-Show history.

This represents an **11.9 percentage-point difference**, indicating that previous No-Show history is associated with a higher likelihood of another No-Show in this dataset.

This finding may be useful for identifying patients who could benefit from additional confirmation or follow-up. However, because the historical fields are synthetic and may not represent a complete real-world patient history, this relationship should be interpreted with caution.

The result is also descriptive and does not establish that previous No-Show behaviour causes future No-Shows.

## KPI 5: No-Show Rate by Distance Band

**Definition:**  
The percentage of appointments resulting in a No-Show across different distance bands from the clinic.

**Why it matters:**  
This KPI examines whether greater distance from the clinic is associated with a higher likelihood of No-Show.

**Linked business question:**  
Q5 — Does distance to the clinic affect the likelihood of attending an appointment?

In [26]:
kpi5_distance = pd.crosstab(
    df['distance_band'],
    df['appointment_outcome'],
    normalize='index'
)['No-Show'].mul(100).round(1)

kpi5_distance

distance_band
<5km      46.5
5-15km    47.3
15km+     54.1
Name: No-Show, dtype: float64

### Result and Interpretation

**Result:**

- **<5 km:** 46.5% No-Show
- **5–15 km:** 47.3% No-Show
- **15 km+:** 54.1% No-Show

The No-Show rate is relatively similar for appointments where patients are located less than 15 km from the clinic. However, the rate increases to **54.1%** among patients located 15 km or more away.

This represents a **7.6 percentage-point difference** between the <5 km and 15 km+ groups.

The result suggests that greater distance from the clinic is associated with a higher No-Show rate in this dataset. This may indicate that distance could be a relevant logistical factor to consider when developing reminder, outreach, or scheduling strategies.

However, the analysis is descriptive and does not establish that distance directly causes patients to miss appointments. The 15 km+ group should also be considered alongside other factors before drawing conclusions.

## Cross-Variable Analysis: Booking Lead Time and Previous No-Show History

To explore whether two factors identified in the individual KPI analysis are related to an even stronger No-Show pattern, booking lead time was examined together with previous No-Show history.

This cross-tabulation compares No-Show rates across combinations of the two variables rather than examining each factor independently.

In [27]:
pd.crosstab(
    [df['lead_time_band'], df['previous_no_show_group']],
    df['appointment_outcome'],
    normalize='index'
)['No-Show'].mul(100).round(1)

lead_time_band         previous_no_show_group
Same week (0-7 days)   No Previous No-Show       21.8
                       Previous No-Show          36.2
2-3 weeks (8-21 days)  No Previous No-Show       34.1
                       Previous No-Show          41.9
3+ weeks (22-60 days)  No Previous No-Show       51.7
                       Previous No-Show          64.1
Name: No-Show, dtype: float64

---

## Combined Analysis

### Combining Lead Time and Previous No-Show History Identifies a Higher-Risk Group

The cross-variable analysis shows that the highest observed No-Show rate occurs when two risk factors appear together.

Patients with a **previous No-Show and an appointment booked 3+ weeks in advance had a 64.1% No-Show rate**. This was higher than both:

- Patients with previous No-Shows but same-week bookings: **36.2%**
- Patients with no previous No-Shows but 3+ week bookings: **51.7%**

This suggests that combining multiple factors can reveal patterns that are not as clear when each variable is examined independently.

From a business perspective, this group could be considered for additional confirmation or reminder strategies in a future intervention. However, this finding should be treated as an observed association rather than a prediction or causal relationship.

---

# Overall Findings & Business Insights

The Week 5 analysis examined appointment No-Show rates across overall attendance, reminder status and channel, booking lead time, previous No-Show history, distance to the clinic, appointment type, gender, and the combined pattern of booking lead time and previous No-Show history.
The results highlight several patterns that may be relevant to HealthConnect's appointment management strategy.

## 1. Booking Lead Time Shows the Strongest Relationship

Booking lead time shows the largest difference in observed No-Show rates among the individual factors examined.

- Same week (0–7 days): **27.8%**
- 2–3 weeks (8–21 days): **37.2%**
- 3+ weeks (22–60 days): **56.9%**

The No-Show rate increased consistently as the time between booking and the appointment became longer. Appointments booked 3+ weeks in advance had a **29.1 percentage-point higher No-Show rate** than appointments booked within the same week.

This suggests that longer booking lead times may be an important factor to consider when designing reminder and appointment-confirmation strategies.

## 2. Previous No-Show History Is Associated With Higher No-Shows

Patients with a previous No-Show had an observed No-Show rate of **55.4%**, compared with **43.5%** among patients with no previous No-Show history.

This represents an **11.9 percentage-point difference**.

The result suggests that previous No-Show history may be useful for identifying patients who are more likely to miss another appointment. However, the historical fields are synthetic and may not represent complete real-world patient histories, so this relationship should be interpreted with caution.

## 3. Reminder Status Shows a Difference in No-Show Rates

Appointments where a reminder was sent had a No-Show rate of **47.4%**, compared with **51.4%** where no reminder was sent.

This represents a **4.0 percentage-point difference**.

The finding suggests an association between reminder status and appointment outcomes in this dataset. However, it does not establish that reminders directly cause patients to attend, as other factors may also differ between the two groups.

## 4. Distance Shows a Smaller but Noticeable Relationship

The No-Show rate was:

- **46.5%** for patients located less than 5 km from the clinic
- **47.3%** for patients located 5–15 km away
- **54.1%** for patients located 15 km or more away

The 15 km+ group had a **7.6 percentage-point higher No-Show rate** than the <5 km group.

This suggests that greater distance may be associated with a higher likelihood of No-Show, although the relationship is less pronounced than the pattern observed for booking lead time.

## 5. Reminder Channel Shows Some Variation

Among appointments where a reminder was sent, the observed No-Show rates were:

- **SMS:** 45.8%
- **Email:** 48.4%
- **WhatsApp:** 49.8%

SMS had the lowest observed No-Show rate among the three channels, while WhatsApp had the highest.

The differences are relatively small, so the results should not be interpreted as evidence that one channel is definitively more effective than another. Further analysis would be required before making changes to the clinic's communication strategy.

## 6. Booking Lead Time and Previous No-Show History Show a Combined Pattern

Examining booking lead time and previous No-Show history together shows that the highest No-Show rates occur when both characteristics are present.

- Same week, no previous no-show: **21.8%**
- Same week, previous no-show: **36.2%**
- 2–3 weeks, no previous no-show: **34.1%**
- 2–3 weeks, previous no-show: **41.9%**
- 3+ weeks, no previous no-show: **51.7%**
- 3+ weeks, previous no-show: **64.1%**

The group with the highest observed No-Show rate — patients with a previous No-Show and a 3+ week booking lead time — reached **64.1%**, compared with **21.8%** for patients with neither characteristic. This represents a **42.3 percentage-point difference**.

Within each lead-time band, patients with a previous No-Show consistently had a higher observed No-Show rate than those without previous No-Shows. This pattern indicates that examining the two characteristics together provides additional insight beyond looking at either factor separately.

As with the individual findings above, this is a descriptive and correlational result based on synthetic data, not a causal claim.

## 7. Booking Lead Time Shows a Larger Observed Difference Than Reminder Status

When the individual KPI results are compared, booking lead time shows a substantially larger difference in No-Show rates than reminder status.

The difference between the 3+ week and same-week booking groups is **29.1 percentage points**, compared with a **4.0 percentage-point difference** between appointments with no reminder and those where a reminder was sent.

This suggests that, within this dataset, booking lead time is a more pronounced observed factor than reminder status. This does not mean that reminders are unimportant; rather, it indicates that the timing of the appointment may deserve greater attention when prioritising areas for further investigation.

## 8. Appointment Type and Gender Show Limited Differences

Appointment type and gender were also examined, but neither showed a strong relationship with No-Show rate.

No-Show rates by appointment type ranged from **46.6%** (General Consultation) to **51.2%** (Follow-up), representing a **4.6 percentage-point spread**. This is relatively small compared with the larger differences observed for booking lead time and previous No-Show history.

No-Show rates by gender were nearly identical between Female (**48.4%**) and Male (**48.7%**). The "Prefer not to say" group showed a lower rate (**43.5%**), but this group contains only 108 records (2.2% of the dataset), so the difference should be interpreted cautiously.

Overall, appointment type and gender do not appear to be strong distinguishing factors for No-Shows in this dataset. Both variables are included as interactive filters on the Power BI dashboard, allowing users to explore these dimensions further.

## Overall Conclusion


The analysis identified several factors associated with appointment No-Shows in the HealthConnect dataset.

The strongest observed relationship was with booking lead time, where No-Show rates increased from 27.8% for same-week bookings to 56.9% for appointments booked 3+ weeks in advance. Previous No-Show history also showed a notable difference, with 55.4% compared with 43.5% among patients without previous No-Shows.

Examining booking lead time and previous No-Show history together revealed an even larger observed gap, with No-Show rates ranging from 21.8% to 64.1% depending on the combination of the two characteristics.

Reminder status, reminder channel, and distance to the clinic showed smaller differences in observed No-Show rates. Appointment type and gender showed limited differences in No-Show rates and are included on the dashboard as exploratory filters rather than key distinguishing factors..

Overall, the findings suggest that booking lead time and previous No-Show history deserve particular attention in the next stage of the project, while reminder and distance patterns should also be considered when interpreting appointment attendance.

These findings provide a useful evidence base for the next stage of the project. The results should be treated as descriptive and correlational, not causal, because the dataset is synthetic and the analysis does not control for other factors.

The findings were used to guide the design of the Week 5 Power BI dashboard, which presents the key No-Show patterns and allows further exploration across relevant appointment characteristics.

---

## Limitations & Considerations

The findings from this analysis should be interpreted within the following limitations and considerations:

1. **Synthetic dataset**  
   The HealthConnect appointment dataset is fictional and synthetic. The overall **48.5% No-Show rate** is a characteristic of this dataset and should not be treated as a real-world clinic benchmark.

2. **Descriptive and correlational analysis**  
   The analysis identifies observed relationships between variables and No-Show outcomes. It does not establish causation. For example, the lower observed No-Show rate among appointments with reminders does not prove that reminders caused patients to attend.

3. **Synthetic historical variables**  
   Variables such as `previous_appointments` and `previous_no_shows` are synthetic and may not represent fully coherent real-world patient histories. Conclusions involving previous No-Show behaviour should therefore be interpreted cautiously.

4. **Missing values**  
   `reminder_channel` contains **1,366 missing values (27.3%)** because these records correspond to appointments where no reminder was sent. This is structural missingness rather than an unknown channel. `distance_to_clinic_km` has **90 missing values (1.8%)**, while `waiting_time_minutes` has **60 missing values (1.2%)**.

5. **Small subgroup sizes**  
   The "Prefer not to say" gender category contains only **108 records (2.2%)**. Its observed No-Show rate should therefore not be given the same weight as the larger gender groups.

6. **No statistical or predictive modelling**  
   This Week 5 analysis is primarily descriptive. No statistical significance testing, predictive modelling, or causal modelling was performed. The observed differences should therefore be treated as patterns for further investigation rather than confirmed effects or predictions.

7. **Synthetic scheduling patterns**  
   The dataset includes characteristics such as Sunday appointments. These may reflect how the synthetic data was generated and should not automatically be interpreted as evidence of a real clinic scheduling issue.

8. **Need for further validation**  
   The findings and recommendations should be tested against real-world HealthConnect data and operational context before being used to make policy, scheduling, reminder, or patient-management decisions.

---

## Power BI Dashboard

The Week 5 findings were translated into an initial Power BI dashboard focused on appointment No-Show patterns.

The dashboard includes:

- Overall No-Show Rate
- No-Show Rate by Booking Lead Time
- No-Show Rate by Previous No-Show History
- No-Show Rate by Distance to Clinic
- No-Show Rate by Reminder Status
- No-Show Rate by Reminder Channel
- Combined Booking Lead Time × Previous No-Show History analysis
- Appointment Type and Gender slicers for interactive exploration

The dashboard was designed to communicate the key patterns identified during the Python analysis and allow users to explore the results across selected appointment characteristics.

## Power BI Dashboard

The Week 5 findings were translated into an initial Power BI dashboard
![Power BI Dashboard](dashboard.png)

